# Architecture — Mejoras futuras recomendadas pero no implementadas

Registro vivo de brechas identificadas durante el ciclo Fases 9–16 (auditoria 2026-08-27) que quedaron
explícitamente fuera de alcance de su fase, o que surgieron como hallazgo secundario mientras se
implementaba otra cosa. Cada entrada indica dónde vive el gap y qué se necesitaría para cerrarlo.

Ver también [kua-unified-management-plan.md](./kua-unified-management-plan.md) para el plan completo por fases.

## 1. Selector de perfil multi-proveedor para Architecture (residual Fase 9)

Al navegar a Architecture sin ninguna aplicación activa ni contexto persistido previamente, el selector
de perfil sigue siendo solo AWS ([App.vue](../../frontend/src/App.vue) plantilla, selector ligado a
`awsProfileId`). Una aplicación Kubernetes-only o GCP/Vercel sin perfil AWS no puede abrirse desde cero
por esta vía (sí funciona si llega con contexto ya persistido o navegando desde Observability).

**Para cerrarlo:** un selector de "perfil de trabajo" multi-proveedor que liste perfiles AWS, GCP,
Vercel y perfiles locales de Kubernetes en un único control, en vez de reutilizar el dropdown AWS.

## 2. Navegación AWS a SQS desde el Canvas (residual Fase 10)

`AwsView.vue` no tiene una pestaña dedicada a SQS (`TABS` no incluye `sqs`), por lo que los nodos SQS
del Canvas no muestran ninguna acción de navegación ("Abrir en vista AWS"), a diferencia de Lambda, EC2,
EventBridge y Step Functions.

**Para cerrarlo:** agregar una pestaña/listado SQS en AwsView (o un modal de detalle standalone) y sumar
`sqs` al mapa `AWS_RESOURCE_TABS` de [App.vue](../../frontend/src/App.vue) y a `AWS_DETAIL_TYPES` en
[ArchitectureCanvas.vue](../../frontend/src/components/architecture/ArchitectureCanvas.vue).

## 3. Señal de salud por recurso AWS (residual Fase 11)

El overlay de salud del Canvas solo puede mostrar `healthy`/`degraded` para nodos Kubernetes, porque
únicamente `lib/kua/kubernetesAdapter.js` calcula `node.health` durante el discovery. Ningún adaptador
AWS produce ese campo, así que los nodos AWS solo pueden mostrar el badge `stale` (sync desactualizado),
nunca degradado/saludable.

**Para cerrarlo:** evaluar salud por recurso individual (no solo a nivel de aplicación como hace hoy
`lib/apm/thresholds.js`) combinando `apm_metric_buckets` (tasa de error, duración, etc. por `resource_id`)
y `apm_collection_cursors` (frescura/última colección por `resource_id`), y proyectar ese resultado sobre
el `node.health` del nodo AWS correspondiente por identidad compartida del registro.

## 4. Fase 13 — Una acción, una revisión

Guardar una operación de Architecture puede disparar `reconcileLinkedApplication()` una segunda vez
(ver [routes/architecture.js](../../routes/architecture.js), llamadas en las líneas ~136, 155, 225, 248 y
329), generando dos revisiones para una sola acción del usuario: la operación original y la proyección/
reconciliación derivada.

**Para cerrarlo:** separar la revisión del usuario de la proyección derivada, o agruparlas en una única
transacción/revisión cuando se originan en la misma request. Auditar los cinco call-sites por el mismo
riesgo de doble escritura.

## 5. Fase 14 — Diagnóstico visible de sincronización

No existe estado persistido de la última reconciliación (éxito, duración, error, recursos/relaciones
divergentes). El usuario debe disparar la reconciliación manualmente desde APM sin ver historial previo.

**Para cerrarlo:** persistir última sincronización exitosa, último error y conteos de divergencia por
aplicación enlazada, y agregar un panel de diagnóstico (APM y/o Architecture) con botón de reintento.

## 6. Fase 15 — Paridad de filtros Canvas ↔ Routes

`ArchitectureRoutes.vue` ya soporta Kubernetes (`Ingress -> Service -> Pod`) pero no tiene controles
propios de proveedor/contexto/namespace; un filtro aplicado en el Canvas no limita automáticamente Routes.

**Para cerrarlo:** compartir el estado de filtro del Canvas (`providerFilter`, `kubeContextFilter`,
`namespaceFilter`) con la vista Routes en vez de duplicar controles de filtro.

## 7. Fase 16 — Análisis de logs observados para sugerir relaciones

Los logs se transmiten y clasifican visualmente en
[useTerminalStreams.js](../../frontend/src/composables/useTerminalStreams.js), pero esa clasificación
nunca se convierte en evidencia persistida o temporal de llamadas HTTP entre Services, nombres DNS
internos, errores recurrentes, correlation IDs o dependencias entre workloads. No existe capa de ML,
y no debería introducirse antes de tener extracción determinista.

**Para cerrarlo:** extracción determinista y sanitizada sobre el stream ya clasificado (sin persistir
payloads crudos ni secretos), con ranking de confianza y las relaciones sugeridas pasando por el mismo
flujo de revisión humana que ya existe para relaciones automáticas/sugeridas.

## 8. GCP y Vercel — discovery Architecture operativo (diferido explícito)

GCP y Vercel siguen como adaptadores planificados en
[ArchitectureView.vue](../../frontend/src/components/architecture/ArchitectureView.vue). El modelo y los
recursos manuales están preparados, pero no hay discovery real para esos proveedores.

**Correcto mantenerlo diferido** hasta cerrar por completo las brechas de AWS y Kubernetes (puntos 1–7
de este documento más las Fases 12–16 del plan).

## 9. Lineage del registro se sobrescribe en vez de fusionarse (hallazgo Fase 12)

`upsertRegistryResource` (ver [applicationRegistryService.js](../../lib/kua/applicationRegistryService.js) y
`ON CONFLICT(identity_key) DO UPDATE` en [database.js](../../lib/apm/database.js)) actualiza
`lineage_json` con el valor de la llamada más reciente. Cuando un mismo recurso canónico se confirma
tanto desde APM como desde Architecture, el `lineage` guardado termina reflejando solo la última fuente
que escribió, no la unión de ambas. El nuevo campo `sources` (Fase 12) sí refleja correctamente ambas
membresías vía `kua_registry_memberships`, pero `lineage` por sí solo puede ser engañoso si se usa como
única fuente de verdad para "de dónde vino este recurso".

**Para cerrarlo:** fusionar (no reemplazar) `lineage` por `identityKey` al hacer upsert, o dejar de
exponer `lineage` en la UI y depender solo de `sources` + las tablas de membership para ese propósito.

## 10. Fase 17 (propuesta, pendiente de confirmación) — provider como identidad de recurso, no de aplicación

Tras el fix de duplicados del 2026-08-27, el usuario propuso generalizar la regla: ningún recurso debe
heredar su `provider` de la aplicación KUA que lo contiene ni de la pestaña activa; siempre debe derivarse
de lo que el recurso realmente es.

**Ya verificado y cumplido sin cambios adicionales:**
- Observability/APM ([ApmObservabilityView.vue](../../frontend/src/components/cloud/apm/ApmObservabilityView.vue))
  nunca incrustó su propio visor de logs: los botones "Abrir logs" solo emiten `open-lambda-logs` /
  `open-kubernetes-logs`, que abren el visor ya existente en la pestaña AWS o Kubernetes. No hay logs
  crudos duplicados entre pestañas.
- El principio "un recurso no hereda el proveedor de su aplicación" ya se generalizó como helper
  reutilizable `resourceOwnProvider()` en
  [applicationRegistryService.js](../../lib/kua/applicationRegistryService.js), usado tanto para el lado
  APM→Architecture como Architecture→APM.

**Pendiente de confirmación explícita del usuario antes de tocar el esquema:**
- ¿`apm_resources.provider` deja de existir como columna, o se mantiene como "proveedor por defecto de
  la aplicación" mientras cada recurso individual gana su propia columna de proveedor autoritativa?
- ¿Se requiere migración de datos para instalaciones existentes, o puede introducirse de forma aditiva
  (como ya ocurrió con el fix de duplicados, que se autocorrige en la próxima reconciliación)?
- ¿Hay algún cambio de navegación pendiente más allá de lo que ya cubre la Fase 10 (acciones de nodo →
  logs/detalle), o el pedido ya queda satisfecho con lo verificado arriba?


## 11. Fase 13 — todavía puede haber 2 revisiones por una acción (mejora parcial)

`ApplicationRegistryService.reconcile()` ahora fusiona sus propias dos mutaciones (proyectar recursos
APM faltantes + sellar ids de correlación del registro) en **una sola** revisión de grafo en vez de dos.
Pero los 5 call sites en [routes/architecture.js](../../routes/architecture.js) (`PUT /graph`,
`POST /operations`, `sync-apply`, `discovery/aws/import`, `snapshots/:id/revert`) primero guardan la
operación propia del usuario (1 revisión) y **después** llaman `reconcileLinkedApplication()`, que puede
producir una revisión derivada adicional. El resultado práctico bajó de "hasta 3 revisiones por una
acción" a "como máximo 2 (una del usuario, una derivada y claramente etiquetada como
`registry.project_apm_resource`/`registry.reconcile`)".

**Para llegar a "siempre exactamente 1"** haría falta que `ArchitectureGraphService.applyOperation` (y
`awsDiscoveryService.applySync`/`importSelection`, y `revertSnapshot`) expongan un modo "calcular sin
guardar", para que `reconcile()` pueda fusionar su proyección directamente en el documento *antes* de que
el llamador haga el único `saveGraph`. Es un refactor de mayor alcance (toca graphService, discoveryService
y la ruta de revert) que no se abordó en esta fase por el riesgo/beneficio frente a lo ya logrado.


## 12. Fase 16 — límites conocidos de las sugerencias de relaciones desde logs

La extracción determinista ([logRelationshipEvidence.js](../../frontend/src/lib/logRelationshipEvidence.js))
ya agrupa errores recurrentes (`extractRecurringErrors`) y recolecta ids de correlación/request/trace
(`extractCorrelationIds`), pero **ninguna de las dos** está todavía conectada a una acción de UI —
`suggestGraphRelationships` solo usa `extractServiceReferences` para proponer relaciones `calls`.

Además:
- Solo compara contra nodos `provider === 'kubernetes'` en el mismo diagrama; no hay correlación
  cruzada con nodos AWS/GCP/Vercel (p. ej. una llamada HTTP a un API Gateway externo no genera sugerencia).
- Requiere que el usuario ya haya abierto el stream de logs de ese Pod/workload (acción "View logs" de
  la Fase 10) antes de poder pedir sugerencias; no hay forma de analizar logs históricos o de un
  workload sin stream activo.
- No hay agrupación de "correlation ID visto en el Service A y en el Service B" para inferir una
  dependencia real de request tracing — eso requeriría cruzar logs de múltiples pods/tabs a la vez,
  fuera de alcance de esta fase.

**Para cerrarlo:** exponer `extractRecurringErrors`/`extractCorrelationIds` en algún panel de
diagnóstico (quizás junto al de la Fase 14), y evaluar una vista que permita elegir cualquier
combinación de tabs de log abiertos (no solo el del nodo seleccionado) para buscar coincidencias de
correlation ID entre servicios.
